In [3]:
import pandas as pd
import numpy as np
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 3.0.3
NumPy version: 2.4.6


In [40]:
dim_company  = pd.DataFrame({
    "company_code": ["C001", "C002", "C003"],
    "company_name": ["Gen Digital Germany GmbH", "Gen Digital CZ s.r.o.", "Gen Digital Austria GmbH"],
    "country": ["DE", "CZ", "AT"],
    "currency": ["EUR", "CZK", "EUR"]
})

print(dim_company)

  company_code              company_name country currency
0         C001  Gen Digital Germany GmbH      DE      EUR
1         C002     Gen Digital CZ s.r.o.      CZ      CZK
2         C003  Gen Digital Austria GmbH      AT      EUR


In [13]:
departments = ["Marketing", "IT", "Finance"]
companies = ["C001", "C002", "C003"]

cc_ids = []
cc_names = []
cc_departments = []
cc_companies = []

for i, company in enumerate(companies, start=1):
    for j, department in enumerate(departments, start=1):
        cc_departments.append(department)
        cc_companies.append(company) 
        cc_names.append(f"{department} {company}")
        cc_ids.append("CC-" + str(i) + "0" + str(j))   

dim_costcenter = pd.DataFrame({
    "cc_id": cc_ids,
    "cc_name": cc_names,
    "department": cc_departments,
    "company_code": cc_companies
})

In [19]:
dim_account = pd.DataFrame({
    "account_code": ["GL-1001", "GL-1002", "GL-1003","GL-2001", "GL-2002", "GL-3001","GL-3002"],
    "account_name": ["Software Licenses", "Salaries", "Travel & Entertainment","Server Hardware", "Office Equipment","Product Revenue", "Service Revenue"],
    "account_type": ["OPEX", "OPEX", "OPEX", "CAPEX", "CAPEX", "REVENUE", "REVENUE"]
})


In [24]:
years = [2024, 2025]
months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]


period_ids = []
fiscal_years = []
fiscal_periods = []
quarters = []
month_names = []

for year in years:
    for period, month in enumerate(months, start=1):
        period_ids.append(str(year) + "-P" + str(period).zfill(2))
        fiscal_years.append(year)
        fiscal_periods.append(period)
        quarters.append("Q" + str((period + 2) // 3))
        month_names.append(month)


In [28]:
dim_date = pd.DataFrame({
    "period_id": period_ids,
    "fiscal_year": fiscal_years,
    "fiscal_period": fiscal_periods,
    "quarter": quarters,
    "month_name": month_names
})


In [30]:
import random
from itertools import product
random.seed(42)

combinations = list(product(
    dim_date["period_id"],
    dim_costcenter["cc_id"],
    dim_account["account_code"]
))

print(len(combinations))

1512


In [38]:
period_ids_fact = []
cc_ids_fact = []
account_codes_fact = []
budgets = []
actuals = []


for period_id, cc_id, account_code in combinations:
    account_type = dim_account.loc[dim_account["account_code"] == account_code, "account_type"].values[0]
    if account_type == "REVENUE":
        budget = random.uniform(1000000, 5000000)
    elif account_type == "CAPEX":
        budget = random.uniform(100000, 1000000)
    else:
        budget = random.uniform(50000, 500000)
    actual = budget * random.uniform(0.85, 1.15)
    period_ids_fact.append(period_id)
    cc_ids_fact.append(cc_id)
    account_codes_fact.append(account_code)
    budgets.append(round(budget, 2))
    actuals.append(round(actual, 2))

print(len(budgets))

fact_finance = pd.DataFrame({
    "period_id": period_ids_fact,
    "cc_id": cc_ids_fact,
    "account_code": account_codes_fact,
    "budget": budgets,
    "actual": actuals
})


1512


In [41]:
df = fact_finance.merge(dim_account, on="account_code", how="left")
df = df.merge(dim_costcenter, on="cc_id", how="left")
df = df.merge(dim_date, on="period_id", how="left")
df = df.merge(dim_company, on="company_code", how="left")

print(df.shape)
df.head()

(1512, 17)


,period_id,cc_id,account_code,budget,actual,account_name,account_type,cc_name,department,company_code,fiscal_year,fiscal_period,quarter,month_name,company_name,country,currency
0,2024-P01,CC-101,GL-1001,272521.82,299767.02,Software Licenses,OPEX,Marketing C001,Marketing,C001,2024,1,Q1,January,Gen Digital Germany GmbH,DE,EUR
1,2024-P01,CC-101,GL-1002,492468.03,454846.19,Salaries,OPEX,Marketing C001,Marketing,C001,2024,1,Q1,January,Gen Digital Germany GmbH,DE,EUR
2,2024-P01,CC-101,GL-1003,151624.69,158352.06,Travel & Entertainment,OPEX,Marketing C001,Marketing,C001,2024,1,Q1,January,Gen Digital Germany GmbH,DE,EUR
3,2024-P01,CC-101,GL-2001,294371.29,251312.28,Server Hardware,CAPEX,Marketing C001,Marketing,C001,2024,1,Q1,January,Gen Digital Germany GmbH,DE,EUR
4,2024-P01,CC-101,GL-2002,319312.18,273405.37,Office Equipment,CAPEX,Marketing C001,Marketing,C001,2024,1,Q1,January,Gen Digital Germany GmbH,DE,EUR


In [50]:
df["variance"] = df["actual"] - df["budget"]
df[["budget", "actual", "variance"]].head()

# Positive variance indicates that the actual amount is greater than the budgeted amount
# a negative variance indicates that the actual amount is less than the budgeted amount

df["variance_pct"] = ((df["variance"] / df["budget"]) * 100).round(2)
df["is_over_budget"] = df["variance"] > 0
df[["budget", "actual", "variance", "variance_pct", "is_over_budget"]].head()

df.to_csv("../data/finance_fact.csv", index=False)
print("Saved successfully")

Saved successfully
